# Aligned-but-Drifting Encoder — Missing Factorial Cell

**The question:** Does an encoder initialized on the correct coordinates
degrade when allowed to drift?

| | Stable | Unstable |
|---|---|---|
| **Aligned** | prescribed ✓ | **THIS EXPERIMENT** |
| **Not aligned** | random_fixed ✓ | free ✓ |

If aligned-but-drifting degrades → drift causally harms even with perfect initialization.
If it stays good → the problem is alignment, not drift.
If it's between prescribed and free → both factors matter independently.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_DIR = '/content/drive/MyDrive/prescribed-axes/paper2_aligned_drifting'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f'Save dir: {SAVE_DIR}')

In [ ]:
import numpy as np, json, time
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split

# === SIGReg ===
class SIGReg(nn.Module):
    def __init__(s, k=17, np_=512):
        super().__init__(); s.np_ = np_
        t = torch.linspace(0, 3, k); dt = 3/(k-1)
        w = torch.full((k,), 2*dt); w[[0,-1]] = dt
        phi = torch.exp(-t.square()/2.0)
        s.register_buffer('t', t); s.register_buffer('phi', phi)
        s.register_buffer('weights', w*phi)
    def forward(s, proj):
        A = torch.randn(proj.size(-1), s.np_, device=proj.device)
        A = A.div_(A.norm(p=2, dim=0))
        x = (proj @ A).unsqueeze(-1) * s.t
        err = (x.cos().mean(-3) - s.phi).square() + x.sin().mean(-3).square()
        return ((err @ s.weights) * proj.size(-2)).mean()

# === Data ===
def synth(n, seed=42):
    rng = np.random.default_rng(seed); eps = []
    for _ in range(n):
        ag = rng.uniform(50, 462, 2).astype(np.float32)
        bp = rng.uniform(100, 412, 2).astype(np.float32)
        ba = np.float32(rng.uniform(0, 2*np.pi))
        st = np.array([ag[0],ag[1],bp[0],bp[1],ba], dtype=np.float32)
        ss, aa = [st.copy()], []
        tgt = rng.uniform(50, 462, 2).astype(np.float32)
        for step in range(300):
            if step % 20 == 0: tgt = rng.uniform(50, 462, 2).astype(np.float32)
            act = np.clip(tgt + rng.normal(0, 10, 2), 0, 512).astype(np.float32)
            d = act - ag; dn = np.linalg.norm(d)
            if dn > 0: ag += d * min(1., 20./dn)
            ag = np.clip(ag, 0, 512)
            tb = bp - ag; cd = np.linalg.norm(tb)
            if 0 < cd < 30:
                f = (30-cd)/30*5; bp += (tb/cd)*f
                ba = (ba + rng.normal(0, .05)*f) % (2*np.pi)
            bp = np.clip(bp, 0, 512)
            st = np.array([ag[0],ag[1],bp[0],bp[1],ba], dtype=np.float32)
            if (step+1) % 5 == 0: ss.append(st.copy()); aa.append(act)
        if len(aa) >= 4:
            eps.append({'s': np.array(ss[:len(aa)+1]), 'a': np.array(aa)})
    return eps

class DS(Dataset):
    def __init__(s, eps, H=3):
        s.w = []
        for e in eps:
            st, a = e['s'], e['a']
            for t in range(len(a)-H):
                s.w.append((st[t:t+H+2].astype(np.float32), a[t:t+H+1].astype(np.float32)))
    def __len__(s): return len(s.w)
    def __getitem__(s, i):
        st, a = s.w[i]
        return torch.from_numpy(st), torch.from_numpy(a)

print('Infrastructure ready')

In [ ]:
# === ENCODERS ===

class PrescribedEncoder(nn.Module):
    """Frozen: extract (x_b, y_b, theta_b), normalize."""
    def __init__(s):
        super().__init__()
        s.register_buffer('sc', torch.tensor([1/512, 1/512, 1/(2*np.pi)]))
    def forward(s, x): return x[..., 2:5] * s.sc


class FreeEncoder(nn.Module):
    """Learned MLP 5->3, random init."""
    def __init__(s):
        super().__init__()
        s.net = nn.Sequential(
            nn.Linear(5, 64), nn.LayerNorm(64), nn.GELU(),
            nn.Linear(64, 64), nn.LayerNorm(64), nn.GELU(),
            nn.Linear(64, 3))
    def forward(s, x): return s.net(x)


class AlignedDriftingEncoder(nn.Module):
    """THE MISSING CELL: initialized to prescribed mapping, then allowed to train.
    
    Starts as h(x) = x[2:5] * scale (same as prescribed).
    But weights are learnable — the encoder can drift away from this initialization.
    
    If it degrades: drift causally harms even with perfect init.
    If it stays good: alignment matters more than stability.
    If it's between: both factors contribute independently.
    """
    def __init__(s):
        super().__init__()
        # Linear layer initialized to extract block coords and scale them
        # Input: [x_a, y_a, x_b, y_b, theta_b] (5D)
        # Output: [x_b/512, y_b/512, theta_b/(2*pi)] (3D)
        s.linear = nn.Linear(5, 3, bias=False)
        
        # Initialize to prescribed mapping
        with torch.no_grad():
            s.linear.weight.zero_()
            s.linear.weight[0, 2] = 1.0 / 512    # x_b
            s.linear.weight[1, 3] = 1.0 / 512    # y_b
            s.linear.weight[2, 4] = 1.0 / (2 * np.pi)  # theta_b
    
    def forward(s, x):
        return s.linear(x)


class AlignedDriftingMLPEncoder(nn.Module):
    """MLP version: same architecture as FreeEncoder, but initialized so that
    the initial output approximates prescribed coordinates.
    
    Uses a 2-phase init:
    1. Train a small regression from 5D input to prescribed output (1 epoch)
    2. Use those weights as initialization, then train normally
    """
    def __init__(s, eps, seed=42):
        super().__init__()
        s.net = nn.Sequential(
            nn.Linear(5, 64), nn.LayerNorm(64), nn.GELU(),
            nn.Linear(64, 64), nn.LayerNorm(64), nn.GELU(),
            nn.Linear(64, 3))
        
        # Pre-train to mimic prescribed output
        sc = torch.tensor([1/512, 1/512, 1/(2*np.pi)])
        
        # Collect some data
        states = []
        for e in eps[:50]:  # use first 50 episodes
            for st in e['s']:
                states.append(torch.from_numpy(st))
        X = torch.stack(states)
        Y = X[:, 2:5] * sc  # prescribed targets
        
        # Quick pre-training
        opt = torch.optim.Adam(s.net.parameters(), lr=1e-3)
        for _ in range(200):
            pred = s.net(X)
            loss = F.mse_loss(pred, Y)
            opt.zero_grad(); loss.backward(); opt.step()
        
        final_loss = F.mse_loss(s.net(X), Y).item()
        print(f'    AlignedDriftingMLP pre-train loss: {final_loss:.6f}')
    
    def forward(s, x): return s.net(x)


class RandomFixedEncoder(nn.Module):
    """Random orthogonal 5->3, normalized, frozen."""
    def __init__(s, seed=0):
        super().__init__()
        s.register_buffer('sc', torch.tensor([1/512, 1/512, 1/512, 1/512, 1/(2*np.pi)]))
        rng = torch.Generator().manual_seed(seed)
        A = torch.randn(5, 3, generator=rng)
        Q, _ = torch.linalg.qr(A)
        s.register_buffer('Q', Q)
    def forward(s, x): return (x * s.sc) @ s.Q


print('Encoders ready')

In [ ]:
# === Model + Training ===

class ActionEncoder(nn.Module):
    def __init__(s):
        super().__init__()
        s.net = nn.Sequential(nn.Linear(2, 32), nn.GELU(), nn.Linear(32, 3))
    def forward(s, a): return s.net(a)

class Predictor(nn.Module):
    def __init__(s):
        super().__init__()
        s.net = nn.Sequential(
            nn.Linear(18, 128), nn.LayerNorm(128), nn.GELU(),
            nn.Linear(128, 128), nn.LayerNorm(128), nn.GELU(),
            nn.Linear(128, 3))
    def forward(s, e, ae):
        return s.net(torch.cat([e, ae], -1).reshape(e.size(0), -1))

class WorldModel(nn.Module):
    def __init__(s, enc, ae, pr, sig):
        super().__init__()
        s.enc, s.ae, s.pr, s.sig = enc, ae, pr, sig
    def forward(s, st, a):
        emb = s.enc(st)
        ctx, tgt = emb[:, :3], emb[:, 3]
        aem = s.ae(a[:, :3])
        p = s.pr(ctx, aem)
        return {'pl': F.mse_loss(p, tgt.detach()),
                'sl': s.sig(emb.transpose(0, 1))}

def train_one_epoch(mdl, tl, opt, lam):
    mdl.train(); tp, ts, n = 0, 0, 0
    for s, a in tl:
        o = mdl(s, a)
        l = o['pl'] + lam * o['sl']
        opt.zero_grad(); l.backward()
        nn.utils.clip_grad_norm_(mdl.parameters(), 1.0)
        opt.step()
        b = s.size(0); tp += o['pl'].item()*b; ts += o['sl'].item()*b; n += b
    return tp/n, ts/n

@torch.no_grad()
def val_loss(mdl, vl):
    mdl.eval(); tp, n = 0, 0
    for s, a in vl:
        o = mdl(s, a); tp += o['pl'].item()*s.size(0); n += s.size(0)
    return tp/n

def run_condition(encoder_type, eps, seed, epochs=30, lam=0.09):
    torch.manual_seed(seed); np.random.seed(seed)
    ds = DS(eps, 3)
    nt = int(len(ds)*0.9); nv = len(ds)-nt
    tr, va = random_split(ds, [nt, nv], generator=torch.Generator().manual_seed(seed))
    tl = DataLoader(tr, batch_size=64, shuffle=True, drop_last=True)
    vl = DataLoader(va, batch_size=64)

    if encoder_type == 'prescribed':
        enc = PrescribedEncoder()
    elif encoder_type == 'free':
        enc = FreeEncoder()
    elif encoder_type == 'aligned_drifting_linear':
        enc = AlignedDriftingEncoder()
    elif encoder_type == 'aligned_drifting_mlp':
        enc = AlignedDriftingMLPEncoder(eps, seed)
    elif encoder_type == 'random_fixed':
        enc = RandomFixedEncoder(seed=seed+1000)
    else:
        raise ValueError(encoder_type)

    mdl = WorldModel(enc, ActionEncoder(), Predictor(), SIGReg())
    trainable = [p for p in mdl.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(trainable, lr=3e-4, weight_decay=1e-3)

    hist = []
    for ep in range(1, epochs+1):
        tp, ts = train_one_epoch(mdl, tl, opt, lam)
        vp = val_loss(mdl, vl)
        hist.append({'ep': ep, 'vp': round(vp,8), 'tp': round(tp,8)})
        if ep in [1,5,10,20,30]:
            print(f'    ep={ep:>2d}  val={vp:.6f}')

    best_vp = min(h['vp'] for h in hist)
    return {'encoder': encoder_type, 'seed': seed,
            'best_val_loss': best_vp, 'final_val_loss': hist[-1]['vp'],
            'history': hist}

print('Training ready')

In [ ]:
# === RUN ALL ===

SEEDS = [42, 123, 777]
N_EP = 200
N_EPOCHS = 30
CONDITIONS = ['prescribed', 'aligned_drifting_linear', 'aligned_drifting_mlp',
              'random_fixed', 'free']

print('='*60)
print('ALIGNED-BUT-DRIFTING — MISSING FACTORIAL CELL')
print('='*60)

R = {}

for seed in SEEDS:
    print(f'\n{"="*40}  SEED {seed}  {"="*40}')
    eps = synth(N_EP, seed)
    print(f'Episodes: {len(eps)}')

    for cond in CONDITIONS:
        print(f'\n  >> {cond}')
        t0 = time.time()
        res = run_condition(cond, eps, seed, N_EPOCHS)
        dt = time.time() - t0
        R[f'{cond}_seed{seed}'] = res
        print(f'  => best={res["best_val_loss"]:.6f}  final={res["final_val_loss"]:.6f}  ({dt:.0f}s)')

    # Autosave
    with open(f'{SAVE_DIR}/results_partial.json', 'w') as f:
        json.dump(R, f, indent=2)
    print(f'  [saved]')

print('\n\nDone.')

In [ ]:
# === RESULTS ===

means = {}
print('='*75)
print('RESULTS')
print('='*75)
print(f'{"Condition":<28} {"Seed 42":>10} {"Seed 123":>10} {"Seed 777":>10} {"Mean":>10}')
print('-'*70)
for cond in CONDITIONS:
    vals = [R[f'{cond}_seed{s}']['best_val_loss'] for s in SEEDS]
    m = np.mean(vals)
    means[cond] = m
    print(f'{cond:<28} {vals[0]:>10.6f} {vals[1]:>10.6f} {vals[2]:>10.6f} {m:>10.6f}')

print(f'\n--- Ratios (vs free) ---')
for cond in CONDITIONS:
    if cond != 'free':
        r = means['free'] / means[cond]
        print(f'{cond:<28} {r:>8.1f}x better than free')

In [ ]:
# === INTERPRETATION ===

print('='*60)
print('FACTORIAL DESIGN — COMPLETE')
print('='*60)

print(f'\n                    Stable          Unstable')
print(f'  Aligned:          prescribed       aligned_drifting_linear')
print(f'                    {means["prescribed"]:.6f}       {means["aligned_drifting_linear"]:.6f}')
print(f'  Not aligned:      random_fixed     free')
print(f'                    {means["random_fixed"]:.6f}       {means["free"]:.6f}')

# Stability effect (holding alignment constant)
stab_aligned = means['aligned_drifting_linear'] / means['prescribed']
stab_unaligned = means['free'] / means['random_fixed']
print(f'\n--- Stability effect ---')
print(f'Among aligned:     {stab_aligned:.1f}x (aligned_drifting / prescribed)')
print(f'Among unaligned:   {stab_unaligned:.1f}x (free / random_fixed)')

# Alignment effect (holding stability constant)
align_stable = means['random_fixed'] / means['prescribed']
align_unstable = means['free'] / means['aligned_drifting_linear']
print(f'\n--- Alignment effect ---')
print(f'Among stable:      {align_stable:.1f}x (random_fixed / prescribed)')
print(f'Among unstable:    {align_unstable:.1f}x (free / aligned_drifting)')

# Independence check
print(f'\n--- Independence check ---')
print(f'If independent: stability effect should be similar across alignment levels')
print(f'  Stability among aligned:   {stab_aligned:.1f}x')
print(f'  Stability among unaligned: {stab_unaligned:.1f}x')
print(f'  Ratio: {max(stab_aligned,stab_unaligned)/max(min(stab_aligned,stab_unaligned),0.001):.1f}x difference')

if abs(stab_aligned - stab_unaligned) / max(stab_aligned, stab_unaligned) < 0.5:
    print('  => Reasonably consistent — factors appear approximately independent')
else:
    print('  => Large difference — factors interact, not independent')

In [ ]:
# === SAVE ===

final = {
    'config': {'seeds': SEEDS, 'episodes': N_EP, 'epochs': N_EPOCHS,
               'conditions': CONDITIONS, 'sigreg_lambda': 0.09,
               'note': 'Aligned-but-drifting: fills missing 2x2 factorial cell'},
    'means': means,
    'factorial': {
        'stable_aligned': means['prescribed'],
        'unstable_aligned_linear': means['aligned_drifting_linear'],
        'unstable_aligned_mlp': means['aligned_drifting_mlp'],
        'stable_unaligned': means['random_fixed'],
        'unstable_unaligned': means['free'],
    },
    'effects': {
        'stability_among_aligned': float(stab_aligned),
        'stability_among_unaligned': float(stab_unaligned),
        'alignment_among_stable': float(align_stable),
        'alignment_among_unstable': float(align_unstable),
    },
    'per_seed': {k: {kk:vv for kk,vv in v.items() if kk!='history'} for k,v in R.items()},
    'full': R,
}
with open(f'{SAVE_DIR}/aligned_drifting_results.json', 'w') as f:
    json.dump(final, f, indent=2)
print(f'Saved to {SAVE_DIR}/aligned_drifting_results.json')

In [ ]:
# === FIGURE ===

import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = {'prescribed':'#2166AC', 'aligned_drifting_linear':'#67A9CF',
          'aligned_drifting_mlp':'#92C5DE',
          'random_fixed':'#F4A582', 'free':'#D6604D'}
labels = {'prescribed':'Prescribed\n(stable+aligned)',
          'aligned_drifting_linear':'Aligned drifting\n(linear init)',
          'aligned_drifting_mlp':'Aligned drifting\n(MLP init)',
          'random_fixed':'Random fixed\n(stable, no align)',
          'free':'Free\n(drifting)'}

# Bar chart
ax = axes[0]
x = range(len(CONDITIONS))
vals = [means[c] for c in CONDITIONS]
stds = [np.std([R[f'{c}_seed{s}']['best_val_loss'] for s in SEEDS]) for c in CONDITIONS]
bars = ax.bar(x, vals, color=[colors[c] for c in CONDITIONS], yerr=stds, capsize=4)
ax.set_xticks(x)
ax.set_xticklabels([labels[c] for c in CONDITIONS], fontsize=7)
ax.set_ylabel('Best validation loss')
ax.set_title('2x2 Factorial: Stability x Alignment', fontweight='bold')
ax.set_yscale('log')
for bar, v in zip(bars, vals):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()*1.5,
            f'{v:.5f}', ha='center', fontsize=6.5)
ax.grid(True, alpha=0.15, axis='y')

# 2x2 factorial heatmap
ax = axes[1]
data = np.array([
    [means['prescribed'], means['aligned_drifting_linear']],
    [means['random_fixed'], means['free']]
])
im = ax.imshow(np.log10(data), cmap='RdYlBu_r', aspect='auto')
ax.set_xticks([0, 1])
ax.set_xticklabels(['Stable', 'Unstable'])
ax.set_yticks([0, 1])
ax.set_yticklabels(['Aligned', 'Not aligned'])
ax.set_title('2x2 Factorial Design (log10 loss)', fontweight='bold')
for i in range(2):
    for j in range(2):
        ax.text(j, i, f'{data[i,j]:.5f}', ha='center', va='center',
                fontsize=10, fontweight='bold',
                color='white' if data[i,j] > 0.001 else 'black')
plt.colorbar(im, ax=ax, label='log10(loss)')

plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/factorial_figure.png', dpi=200)
plt.show()
print('Figure saved')